In [1]:
import langchain
from extract4chem_fluorine.tools import processDoc,extract_doi
from pathlib import Path
import os


In [2]:
markdown_text = Path("../data/raw/full.md").read_text(encoding="utf-8")

In [3]:
extract_doi(markdown_text)

'10.1016/j.apcatb.2018.11.023'

In [4]:
markdown_text

'# Modulation of b-axis thickness within MFI zeolite: Correlation with variation of product diffusion and coke distribution in the methanol-to-hydrocarbons conversion\n\n![](images/8232d2265168978010a334c2cbd8812eff8d422229e6e0ff9dea744305852702.jpg)\n\nNing Wang $^{a,b,*1}$ , Yilin Hou $^{a,1}$ , Wenjing Sun $^{c}$ , Dali Cai $^{a}$ , Zhaohui Chen $^{a}$ , Lingmei Liu $^{b}$ , Binghui Ge $^{d}$ , Ling Hu $^{a}$ , Weizhong Qian $^{a,*}$ , Fei Wei $^{a}$\n\na Beijing Key Laboratory of Green Chemical Reaction Engineering and Technology, Department of Chemical Engineering, Tsinghua University, Beijing 100084, China  \n<sup>b</sup> Advanced Membranes and Porous Materials Center, Physical Sciences and Engineering Division, King Abdullah University of Science and Technology (KAUST), Thuwal 23955-6900, Saudi Arabia  \n<sup>c</sup> China-America Cancer Research Institute, Key Laboratory for Medical Molecular Diagnostics of Guangdong Province, Guangdong Medical University, Dongguan, Guangdong 5

In [5]:
splited_texts = processDoc(markdown_text)
splited_texts[:2]

[{'title_level': 'H1',
  'content_title': 'Modulation of b-axis thickness within MFI zeolite: Correlation with variation of product diffusion and coke distribution in the methanol-to-hydrocarbons conversion',
  'content': '![](images/8232d2265168978010a334c2cbd8812eff8d422229e6e0ff9dea744305852702.jpg)  \nNing Wang $^{a,b,*1}$ , Yilin Hou $^{a,1}$ , Wenjing Sun $^{c}$ , Dali Cai $^{a}$ , Zhaohui Chen $^{a}$ , Lingmei Liu $^{b}$ , Binghui Ge $^{d}$ , Ling Hu $^{a}$ , Weizhong Qian $^{a,*}$ , Fei Wei $^{a}$  \na Beijing Key Laboratory of Green Chemical Reaction Engineering and Technology, Department of Chemical Engineering, Tsinghua University, Beijing 100084, China\n<sup>b</sup> Advanced Membranes and Porous Materials Center, Physical Sciences and Engineering Division, King Abdullah University of Science and Technology (KAUST), Thuwal 23955-6900, Saudi Arabia\n<sup>c</sup> China-America Cancer Research Institute, Key Laboratory for Medical Molecular Diagnostics of Guangdong Province, Gu

# 准备数据

In [6]:
# title abstract experiment
main_input = {
    "title": f"# {splited_texts[0]["content_title"]}",
    "abstract": "",
    "experiment": ""
}
abstract_info = []
exp_infos = []
index = 0
while index<=len(splited_texts)-1:
    if splited_texts[index]["content_title"].upper() in ["ARTICLEINFO", "ABSTRACT"]:
        abstract_info.append(f"# {splited_texts[index]["content_title"]}\n{splited_texts[index]["content"]}")
    elif "EXPERIMENTAL" in splited_texts[index]["content_title"].upper():
        raw_exp_title = splited_texts[index]["content_title"]
        exp_infos.append(f"# {splited_texts[index]["content_title"]}\n{splited_texts[index]["content"]}")
        break
    index += 1



for block in splited_texts[index+1:]:
    if not os.path.commonprefix([block["content_title"].strip(), raw_exp_title.strip()]):
        break
    exp_infos.append(f"# {block["content_title"]}\n{block["content"]}")
main_input["experiment"] = "\n\n".join(exp_infos)
main_input["abstract"] = "\n\n".join(abstract_info)

In [7]:
main_input

{'title': '# Modulation of b-axis thickness within MFI zeolite: Correlation with variation of product diffusion and coke distribution in the methanol-to-hydrocarbons conversion',
 'abstract': '# ARTICLEINFO\nKeywords:  \nMFI zeolite  \nb-axis thickness  \nAnisotropy  \nCoke formation  \nMethanol-to-hydrocarbons\n\n# ABSTRACT\nThe conversion of methanol-to-hydrocarbons (MTH) has been studied over a series of  $\\mathrm{Zn / ZSM - 5}$  zeolites with different thickness of b-axis, as well as similar lengths of a-axis. It has been demonstrated that the decrease of b-axis thickness from  $220\\mathrm{nm}$  to  $2\\mathrm{nm}$  leads to the remarkably longer lifetime, accompanied by the shift of selectivity toward trimethylbenzene and increased coke tolerance capacity. Methylbenzenes, as the intermediate product of the aromatic-based cycle, can diffuse out of the straight channels in the  $\\mathrm{Zn / ZSM - 5}$  nanosheet quickly, suppressing the aromatic-based cycle. The evolution of coke

In [8]:
start_index = 0
end_index = len(splited_texts)
end_flags = ["acknowledgement", "references", "appendix"]
for i, t in enumerate(splited_texts):
    if "abstract" in t["content_title"].lower():
        start_index = i
    elif any(flag in t["content_title"].lower() for flag in end_flags) and i > start_index:
        end_index = i
        break
eff_texts = splited_texts[start_index:end_index]
eff_paper = "\n\n".join([f"# {t["content_title"]}\n{t["content"]}" for t in eff_texts])

In [9]:
eff_paper

'# ABSTRACT\nThe conversion of methanol-to-hydrocarbons (MTH) has been studied over a series of  $\\mathrm{Zn / ZSM - 5}$  zeolites with different thickness of b-axis, as well as similar lengths of a-axis. It has been demonstrated that the decrease of b-axis thickness from  $220\\mathrm{nm}$  to  $2\\mathrm{nm}$  leads to the remarkably longer lifetime, accompanied by the shift of selectivity toward trimethylbenzene and increased coke tolerance capacity. Methylbenzenes, as the intermediate product of the aromatic-based cycle, can diffuse out of the straight channels in the  $\\mathrm{Zn / ZSM - 5}$  nanosheet quickly, suppressing the aromatic-based cycle. The evolution of coke species, including the quantity, types, and location, as a function of the reaction time, has been systematically investigated. During the initial reaction period, the coke preferentially forms in mesopores and then is deposited mainly in micropores as the reaction proceeds. The  $\\mathrm{Zn / ZSM - 5}$  nanoshe

# 准备模型

In [10]:
DOUBAO_config = {
    "model": "doubao-seed-1-6-flash",
    "model_provider": "openai",
    "temperature": 0,
    "base_url": os.getenv("DOUBAO_BASEURL"),        # 这里替换为自己的 Base URL
    "api_key": os.getenv("DOUBAO_APIKEY_FS"),       # 这里替换为自己的 Key
    "max_completion_tokens" : 64000
}

gpt_config = {
    "model": "gpt-5",
    "model_provider": "openai",
    "temperature": 0,
    "base_url": os.getenv("BASEURL_FS"),        # 这里替换为自己的 Base URL
    "api_key": os.getenv("OPENAI_APIKEY_FS"),       # 这里替换为自己的 Key
    "reasoning_effort" : "low"
}

qwen_stream_config = {
    "model": "qwen3-max-preview",
    "model_provider": "openai",
    "temperature": 0,
    "base_url": os.getenv("QWEN_BASEURL"),        # 这里替换为自己的 Base URL
    "api_key": os.getenv("QWEN_APIKEY"),       # 这里替换为自己的 Key
    # "response_format" : {
    #     "type": "json_object"
    # },
    "extra_body":{"enable_thinking":True},
    # 流式输出方式调用
    "stream" : True,
}
qwen_config = {
    "model": "qwen3-max",
    "model_provider": "openai",
    "temperature": 0,
    "base_url": os.getenv("QWEN_BASEURL"),        # 这里替换为自己的 Base URL
    "api_key": os.getenv("QWEN_APIKEY"),       # 这里替换为自己的 Key
    "response_format" : {
        "type": "json_object"
    },
}

In [11]:
from langchain.chat_models import init_chat_model
repair_llm = init_chat_model(**DOUBAO_config)

In [12]:

main_llm = init_chat_model(**qwen_stream_config)

/tmp/ipykernel_1237471/2899556767.py:1: UserWarning: WARNING! stream is not default parameter.
                stream was transferred to model_kwargs.
                Please confirm that stream is what you intended.
  main_llm = init_chat_model(**qwen_stream_config)


In [ ]:
# 模型验证
for x in main_llm.stream("你是谁，用json回答"):
    if x.content.strip() == "":
        continue
    print(x.content, end="\n", flush=True)

NameError: name 'main_llm' is not defined

In [14]:
from langchain_core.prompts import ChatPromptTemplate
prompt_temp = ChatPromptTemplate.from_template(Path("../prompts/主信号抽取.txt").read_text(encoding="utf-8"))
prompt_temp.pretty_print()

================================ Human Message =================================

<System>
任务：从【标题/摘要/实验】中抽取“用于唯一指代样品/材料/催化剂”的命名（material_id），并同时抽取 IZA 框架代码（framework_code）。只输出严格 JSON。

规则（material_id）：
- 仅在提供文本内抽取；不得臆测；宁缺毋滥。
- 计作样品名：能唯一指代实验样品的标签（如 Sample A/B/C、Compound 1/2、M1–M3、Zn/ZSM-5-2 wt%、UiO-66-NH2、Si/Al=40 等）。
- 排除通用类名/试剂/溶剂/反应物（polymer、catalyst、ethanol、H2SO4、模板剂等），除非该名称在文中明确作为样品标签用于表征/应用。
- 若出现系列/枚举（如 “x=0,1,2 wt%” 或 “M1–M3”），必须**展开为具体样品**；系列统称本身（如 Zn/Z5(x,y)）**不单列**。
- **禁止占位符**：id 不得含 x、y、?、* 等变量或占位形式；**id 必须为文本中真实出现**的具体写法，或**由‘桥接合成步骤’在文中明示的关系**接组合得到；任何仅凭推断得到的写法一律不得作为 id。
- **桥接合成步骤**：若文本明确说明“将一组变体统一进行某改性/负载以得到目标催化剂”，可据此组合出具体催化剂名称（如由“ZSM-5 nanosheet (2 nm)”统一负载 Zn 得 “Zn/ZSM-5 (2 nm)”），并将基体写入 aliases；若无此明确说明，**禁止组合**，不要把基体名放入催化剂的 aliases。
- 去重：忽略大小写与多余空格后 id 不得重复；id 不能出现在自身 aliases 中；aliases 去重。

规则（why）：
- 说明当前同级键为何会取当前值，给出来源与短引
- 若某一元素为空，**必须简要说明为空的原因**，


规则（framework_code与paper_frameworks，IZA 三字母）：
- 有效格式：**三位大写字母**（MFI/BEA/FAU/MOR/CHA/FER/MWW/LTA…）。
- 在以下上下文之一即视为有效：
  1

In [15]:
from extract4chem_fluorine.entities.material_ID import Er4MaterialId
Er4MaterialId.model_json_schema()

{'$defs': {'MaterialID': {'properties': {'id': {'description': '材料/样品命名（可含字母、数字、符号），保留论文原样',
     'minLength': 1,
     'title': 'Id',
     'type': 'string'},
    'aliases': {'description': '同义/简称，不允许重复，不允许和id相同',
     'items': {'type': 'string'},
     'title': 'Aliases',
     'type': 'array'},
    'framework_code': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'description': 'IZA 的三字母骨架代码',
     'title': 'Framework Code'},
    'why': {'description': '取值理由说明，≤120字', 'title': 'Why', 'type': 'string'}},
   'required': ['id', 'why'],
   'title': 'MaterialID',
   'type': 'object'}},
 'properties': {'material_ids': {'items': {'$ref': '#/$defs/MaterialID'},
   'title': 'Material Ids',
   'type': 'array'},
  'paper_frameworks': {'description': '论文级出现但无法绑定到具体样品的IZA 三字母骨架代码列表',
   'items': {'type': 'string'},
   'title': 'Paper Frameworks',
   'type': 'array'},
  'why': {'description': '取值理由说明，≤300字', 'title': 'Why', 'type': 'string'}},
 'required': ['why'],
 'titl

In [16]:
from extract4chem_fluorine.tools import MyJSONParser

In [17]:
chain = prompt_temp | main_llm  | MyJSONParser(Er4MaterialId)#.with_structured_output(Er4MaterialId, include_raw = True , method = "function_calling")

In [18]:
res = chain.stream(main_input)

In [19]:
Er4MaterialId

extract4chem_fluorine.entities.material_ID.Er4MaterialId

In [20]:
main_input

{'title': '# Modulation of b-axis thickness within MFI zeolite: Correlation with variation of product diffusion and coke distribution in the methanol-to-hydrocarbons conversion',
 'abstract': '# ARTICLEINFO\nKeywords:  \nMFI zeolite  \nb-axis thickness  \nAnisotropy  \nCoke formation  \nMethanol-to-hydrocarbons\n\n# ABSTRACT\nThe conversion of methanol-to-hydrocarbons (MTH) has been studied over a series of  $\\mathrm{Zn / ZSM - 5}$  zeolites with different thickness of b-axis, as well as similar lengths of a-axis. It has been demonstrated that the decrease of b-axis thickness from  $220\\mathrm{nm}$  to  $2\\mathrm{nm}$  leads to the remarkably longer lifetime, accompanied by the shift of selectivity toward trimethylbenzene and increased coke tolerance capacity. Methylbenzenes, as the intermediate product of the aromatic-based cycle, can diffuse out of the straight channels in the  $\\mathrm{Zn / ZSM - 5}$  nanosheet quickly, suppressing the aromatic-based cycle. The evolution of coke

In [21]:
%load_ext autoreload
%autoreload 2

In [22]:
prompt_temp.pretty_print()

================================ Human Message =================================

<System>
任务：从【标题/摘要/实验】中抽取“用于唯一指代样品/材料/催化剂”的命名（material_id），并同时抽取 IZA 框架代码（framework_code）。只输出严格 JSON。

规则（material_id）：
- 仅在提供文本内抽取；不得臆测；宁缺毋滥。
- 计作样品名：能唯一指代实验样品的标签（如 Sample A/B/C、Compound 1/2、M1–M3、Zn/ZSM-5-2 wt%、UiO-66-NH2、Si/Al=40 等）。
- 排除通用类名/试剂/溶剂/反应物（polymer、catalyst、ethanol、H2SO4、模板剂等），除非该名称在文中明确作为样品标签用于表征/应用。
- 若出现系列/枚举（如 “x=0,1,2 wt%” 或 “M1–M3”），必须**展开为具体样品**；系列统称本身（如 Zn/Z5(x,y)）**不单列**。
- **禁止占位符**：id 不得含 x、y、?、* 等变量或占位形式；**id 必须为文本中真实出现**的具体写法，或**由‘桥接合成步骤’在文中明示的关系**接组合得到；任何仅凭推断得到的写法一律不得作为 id。
- **桥接合成步骤**：若文本明确说明“将一组变体统一进行某改性/负载以得到目标催化剂”，可据此组合出具体催化剂名称（如由“ZSM-5 nanosheet (2 nm)”统一负载 Zn 得 “Zn/ZSM-5 (2 nm)”），并将基体写入 aliases；若无此明确说明，**禁止组合**，不要把基体名放入催化剂的 aliases。
- 去重：忽略大小写与多余空格后 id 不得重复；id 不能出现在自身 aliases 中；aliases 去重。

规则（why）：
- 说明当前同级键为何会取当前值，给出来源与短引
- 若某一元素为空，**必须简要说明为空的原因**，


规则（framework_code与paper_frameworks，IZA 三字母）：
- 有效格式：**三位大写字母**（MFI/BEA/FAU/MOR/CHA/FER/MWW/LTA…）。
- 在以下上下文之一即视为有效：
  1

In [23]:
results = []
for x in chain.stream(main_input):
    results.append(x)

In [24]:
len(results)

24

In [26]:
results

[ParseResult(parse=Er4MaterialId(material_ids=[MaterialID(id='Zn/ZSM-5 nanosheet', aliases=['ZSM-5 nanosheet (2 nm thickness)'], framework_code=None, why="在摘要中作为唯一指代2nm厚度样品的标签用于表征扩散行为和结焦分布（短引：'The Zn/ZSM-5 nanosheet shows a crystal face dependency on coke deposition'）；基体名称来自实验部分制备步骤（'ZSM-5 nanosheet (2 nm thickness)'），但无近邻MFI代码绑定证据。"), MaterialID(id='conventional Zn/ZSM-5 catalyst', aliases=[], framework_code=None, why="在摘要中作为唯一指代220nm厚度对照样品的标签用于讨论结焦密度差异（短引：'for the conventional Zn/ZSM-5 catalyst, the diffusion of small molecule products is through both channels'）；无近邻MFI代码绑定证据，且无明确基体名称用于aliases。")], paper_frameworks=['MFI'], why=''), raw_text='{\n  "material_ids": [\n    {\n      "id": "Zn/ZSM-5 nanosheet",\n      "aliases": ["ZSM-5 nanosheet (2 nm thickness)"],\n      "framework_code": null,\n      "why": "在摘要中作为唯一指代2nm厚度样品的标签用于表征扩散行为和结焦分布（短引：\'The Zn/ZSM-5 nanosheet shows a crystal face dependency on coke deposition\'）；基体名称来自实验部分制备步骤（\'ZSM-5 nanosheet (2 nm thickness)\'），但无近邻MFI代码绑定证

In [27]:
results[-1].model_dump(mode = "json")["parse"]

{'material_ids': [{'id': 'Zn/ZSM-5 nanosheet',
   'aliases': ['ZSM-5 nanosheet (2 nm thickness)'],
   'framework_code': None,
   'why': "在摘要中作为唯一指代2nm厚度样品的标签用于表征扩散行为和结焦分布（短引：'The Zn/ZSM-5 nanosheet shows a crystal face dependency on coke deposition'）；基体名称来自实验部分制备步骤（'ZSM-5 nanosheet (2 nm thickness)'），但无近邻MFI代码绑定证据。"},
  {'id': 'conventional Zn/ZSM-5 catalyst',
   'aliases': [],
   'framework_code': None,
   'why': "在摘要中作为唯一指代220nm厚度对照样品的标签用于讨论结焦密度差异（短引：'for the conventional Zn/ZSM-5 catalyst, the diffusion of small molecule products is through both channels'）；无近邻MFI代码绑定证据，且无明确基体名称用于aliases。"}],
 'paper_frameworks': ['MFI'],
 'why': "1. material_ids仅抽取摘要中明确作为实验样品标签且用于表征的条目（'Zn/ZSM-5 nanosheet'对应2nm厚度，'conventional Zn/ZSM-5 catalyst'对应220nm厚度）；10nm和60nm厚度样品无独立标签（如具体命名或系列展开），故不抽取。2. paper_frameworks取自标题'# Modulation of b-axis thickness within MFI zeolite'中'MFI zeolite'关键词，符合论文级框架代码规则。"}

In [143]:
texts = ""
for r in results:
    texts += r.content

In [146]:
!pip install json-repair
from json_repair import repair_json

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/


In [150]:
print(repair_json(texts))

{"material_ids": [{"id": "Zn/Z5(2,50)", "aliases": ["Zn/ZSM-5 nanosheet"], "why": "Abbreviation 'Zn/Z5(x,y)' explicitly defined in experimental section with x=b-axis thickness and y=Si/Al ratio. For 2nm thickness sample, gel composition '100SiO2:1Al2O3' implies Si/Al atomic ratio=50. Abstract refers to this sample as 'Zn/ZSM-5 nanosheet' for nanosheet morphology."}, {"id": "Zn/Z5(10,50)", "aliases": [], "why": "Abbreviation 'Zn/Z5(x,y)' defined in experimental section. For 10nm thickness sample, composition '100SiO2:1Al2O3' implies Si/Al ratio=50. No specific aliases found in text; only generic references."}, {"id": "Zn/Z5(60,50)", "aliases": [], "why": "Abbreviation 'Zn/Z5(x,y)' defined in experimental section. For 60nm thickness sample, composition '100TEOS:1Al2O3' implies Si/Al ratio=50. No specific aliases; referred to generically in characterization."}, {"id": "Zn/Z5(220,50)", "aliases": ["conventional Zn/ZSM-5 catalyst"], "why": "Abbreviation 'Zn/Z5(x,y)' defined in experimental 

# 结果序列化

In [73]:
outpath = Path("../data/out/extraction_results.json")

In [74]:
import orjsonl
orjsonl.save(outpath, [result.model_dump(mode = "json")])